# AI 3 · 2주차 정보 추출·날짜 정규화·위험 판단

OCR 결과에서 **제품명·소비기한·알레르기**를 뽑고, 날짜를 정규화하고, 사용자 알레르기와 맞춰 위험도를 정합니다.
AI 2가 찍은 실제 사진(매운새우깡)의 OCR 결과로 정확도를 재고, AI 4 안내 문장까지 이어지는지 확인합니다.

**실행 방법:** 메뉴에서 **런타임 → 모두 실행**을 누르면 됩니다. GPU는 필요 없고 **1~2분** 걸립니다.

## 1. 레포 받아오기

GitHub에서 코드를 고쳤다면 이 셀부터 다시 실행해야 바뀐 내용이 반영됩니다.

In [ ]:
REPO_URL = "https://github.com/DL-2026F-Team05/package-info-extraction.git"

import os, subprocess, sys

os.chdir("/content")
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")   # 비공개 레포일 때만 필요
except Exception:
    token = None
url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if not os.path.exists("package-info-extraction"):
    if subprocess.run(["git", "clone", "-q", url, "package-info-extraction"]).returncode != 0:
        raise RuntimeError("레포를 받아오지 못했습니다. REPO_URL과 공개/비공개 설정(토큰)을 확인하세요.")
os.chdir("/content/package-info-extraction")
subprocess.run(["git", "pull", "-q"])
sys.path.insert(0, os.getcwd())
print("현재 위치:", os.getcwd())

## 2. 패키지 설치

`pydantic`은 코랩에 이미 들어 있어서 보통 바로 끝납니다.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("설치 완료")

## 3. 테스트

날짜 형식, 알레르기 매칭, 위험 판단, 실제 사진 OCR 결과를 확인합니다. 모두 `passed`가 나와야 합니다.

In [ ]:
!python -m pytest -q

## 4. AI 2 OCR 결과로 정보 추출

AI 2 `run_pipeline` 결과 JSON을 AI 2 문서에 적힌 대로 `extract(result["ocr"], user_allergens)`에 넣습니다.
`USER_ALLERGENS`를 바꿔 보세요.

In [ ]:
import json
from pathlib import Path
from extraction import extract
from extraction.__main__ import summary

USER_ALLERGENS = ["우유", "땅콩"]
TODAY = "2026-11-17"

for f in sorted(Path("data/samples/ai2_week2").glob("*.json")):
    result = json.loads(f.read_text(encoding="utf-8"))
    info = extract(result["ocr"], USER_ALLERGENS, TODAY)
    info["image"] = f.name
    print(summary(info), end="\n\n")

## 5. 평가

정답 라벨(`data/labels/labels.csv`)과 비교해 항목별 정확도, F1, 위험 오분류율을 계산합니다.
결과는 `results/ai3_week2/`에 저장됩니다.

In [ ]:
from IPython.display import Markdown, display
from extraction.evaluate import main as evaluate

evaluate(["--ocr-dir", "data/samples/ai2_week2", "--labels", "data/labels/labels.csv",
          "--today", TODAY, "--out", "results/ai3_week2"])
display(Markdown(Path("results/ai3_week2/summary.md").read_text(encoding="utf-8")))

## 6. AI 4와 연결 (안내 문장까지)

AI 4 레포(AI_tts_web)를 받아 와서, AI 3 결과를 AI 4 안내 문장(`speech.compose`)에 그대로 넣어 봅니다.
마지막 줄은 AI_tts_web 의 테스트를 이 모듈로 돌려 보는 확인입니다 (AI_tts_web 파일은 바꾸지 않고 복사본에서만).

In [ ]:
AI4 = "/content/AI_tts_web"
if not os.path.exists(AI4):
    subprocess.run(["git", "clone", "-q", "https://github.com/DL-2026F-Team05/AI_tts_web.git", AI4], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{AI4}/requirements.txt", "-r", f"{AI4}/requirements-dev.txt"])

env = dict(os.environ, PYTHONPATH=f"{AI4}:{os.getcwd()}")
subprocess.run([sys.executable, "-m", "extraction.demo", "--today", TODAY, "--out", "results/ai3_week2/demo.md"],
               env=env, check=True, capture_output=True)
display(Markdown(Path("results/ai3_week2/demo.md").read_text(encoding="utf-8")))

r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_integration.py", "-rA"],
                   env=dict(os.environ, AI_TTS_WEB=AI4), capture_output=True, text=True)
print(r.stdout[-1500:])

## 7. 직접 넣어 보기

글자를 줄바꿈으로 나눠 넣으면 OCR 결과 대신 쓸 수 있습니다.

In [ ]:
text = """제품명: 고소한 두유
원재료명: 정제수, 대두(국산), 설탕
이 제품은 땅콩, 호두를 사용한 제품과 같은 제조시설에서 제조하고 있습니다.
소비기한 26.11.20까지"""

print(summary(extract(text, ["땅콩"], TODAY)))